In [11]:
# Cell 1 — Imports + seed + device

import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    roc_auc_score, average_precision_score, confusion_matrix
)

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed = 42
seed_everything(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)
print("torch =", torch.__version__)


device = cuda
torch = 2.5.1+cu121


In [12]:
# Cell 2 — Load CSV + choose features/label + basic sanity checks

DATA_PATH = "../qnn_dataset/qnn_dataset_v2/qnn_features_v2.csv"
df = pd.read_csv(DATA_PATH)

LABEL_COL = "label"
GROUP_COL = "vul_id"  # để tránh leakage giữa (vulnerable/fixed) cùng CVE/vul_id

# Chọn toàn bộ cột số làm feature, trừ label
num_cols = df.select_dtypes(exclude=["object"]).columns.tolist()
feature_cols = [c for c in num_cols if c != LABEL_COL]

X = df[feature_cols].to_numpy(dtype=np.float32)
y = df[LABEL_COL].to_numpy(dtype=np.int64)
groups = df[GROUP_COL].astype(str).to_numpy()

print("df.shape =", df.shape)
print("n_features =", X.shape[1])
print("label distribution:", dict(pd.Series(y).value_counts().sort_index()))
print("example feature cols:", feature_cols[:10])


df.shape = (1578, 142)
n_features = 136
label distribution: {0: np.int64(1169), 1: np.int64(409)}
example feature cols: ['ast_n_nodes', 'ast_n_edges', 'ast_n_leaves', 'ast_max_depth', 'ast_avg_branching', 'ast_error_nodes', 'ast_truncated', 'parse_ok', 'feat_0', 'feat_1']


In [13]:
# Cell 3 — Group-stratified split (train/val/test) to reduce leakage risk

def make_group_stratified_splits(X, y, groups, seed=42):
    """
    Tạo 1 split train/val/test theo kiểu stratified + group.
    Cách làm: dùng StratifiedGroupKFold 5-fold:
      - fold 0 làm test
      - trên phần còn lại, fold 0 (của 4-fold) làm val
    """
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    splits = list(sgkf.split(X, y, groups))
    trainval_idx, test_idx = splits[0]

    X_tv, y_tv, g_tv = X[trainval_idx], y[trainval_idx], groups[trainval_idx]

    sgkf2 = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=seed)
    splits2 = list(sgkf2.split(X_tv, y_tv, g_tv))
    train_idx_tv, val_idx_tv = splits2[0]

    train_idx = trainval_idx[train_idx_tv]
    val_idx = trainval_idx[val_idx_tv]

    return train_idx, val_idx, test_idx

train_idx, val_idx, test_idx = make_group_stratified_splits(X, y, groups, seed=seed)

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

print("train/val/test =", len(train_idx), len(val_idx), len(test_idx))
print("train pos% =", y_train.mean().round(4), "val pos% =", y_val.mean().round(4), "test pos% =", y_test.mean().round(4))


train/val/test = 952 293 333
train pos% = 0.2563 val pos% = 0.198 test pos% = 0.3213


In [14]:
# Cell 4 — Standardization (fit on train only)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train).astype(np.float32)
X_val_s   = scaler.transform(X_val).astype(np.float32)
X_test_s  = scaler.transform(X_test).astype(np.float32)


In [15]:
# Cell 5 — Torch Dataset/DataLoader (+ WeightedRandomSampler for imbalance)

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

class TabDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)  # BCE wants float
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = TabDataset(X_train_s, y_train)
val_ds   = TabDataset(X_val_s, y_val)
test_ds  = TabDataset(X_test_s, y_test)

# sampler: tăng tần suất lấy positive để giảm bias
class_counts = np.bincount(y_train)
w0 = 1.0 / max(class_counts[0], 1)
w1 = 1.0 / max(class_counts[1], 1)
sample_weights = np.where(y_train == 1, w1, w0).astype(np.float32)

sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

BATCH_SIZE = 64

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, drop_last=False)
val_loader   = DataLoader(val_ds, batch_size=256, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

print("class_counts(train) =", class_counts)


class_counts(train) = [708 244]


In [16]:
# Cell 6 — Models: (A) MLP, (B) 1D-CNN, (C) Feature-Transformer

class MLP(nn.Module):
    def __init__(self, d_in: int, hidden=(256, 128, 64), dropout=0.2):
        super().__init__()
        layers = []
        prev = d_in
        for h in hidden:
            layers += [
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout),
            ]
            prev = h
        self.backbone = nn.Sequential(*layers)
        self.head = nn.Linear(prev, 1)  # logits
    def forward(self, x):
        z = self.backbone(x)
        return self.head(z).squeeze(-1)

class CNN1D(nn.Module):
    """
    Treat features as a 1D "signal" length d_in, channel=1.
    """
    def __init__(self, d_in: int, channels=(32, 64, 128), dropout=0.2):
        super().__init__()
        c1, c2, c3 = channels
        self.conv = nn.Sequential(
            nn.Conv1d(1, c1, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv1d(c1, c2, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2),
            nn.Conv1d(c2, c3, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool1d(1),
        )
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(c3, 1)
        self.d_in = d_in

    def forward(self, x):
        x = x.unsqueeze(1)            # (B, 1, d_in)
        z = self.conv(x).squeeze(-1)  # (B, c3)
        z = self.dropout(z)
        return self.head(z).squeeze(-1)

class FeatureTokenizer(nn.Module):
    """
    Tokenize each scalar feature into a d_model vector:
      token_i = W_i * x_i + b_i  (learnable per-feature)
    """
    def __init__(self, n_features: int, d_model: int):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(n_features, d_model) * 0.02)
        self.bias   = nn.Parameter(torch.zeros(n_features, d_model))

    def forward(self, x):
        # x: (B, F)
        # out: (B, F, D)
        return x.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0)

class TabTransformer(nn.Module):
    """
    Simple TransformerEncoder over per-feature tokens.
    Uses CLS token pooling.
    """
    def __init__(self, n_features: int, d_model=128, nhead=8, nlayers=2, dropout=0.1):
        super().__init__()
        self.tok = FeatureTokenizer(n_features, d_model)
        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos = nn.Parameter(torch.randn(1, n_features + 1, d_model) * 0.01)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=4*d_model,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=nlayers)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x):
        B, F = x.shape
        tok = self.tok(x)                         # (B, F, D)
        cls = self.cls.expand(B, 1, -1)           # (B, 1, D)
        z = torch.cat([cls, tok], dim=1)          # (B, 1+F, D)
        z = z + self.pos[:, : (F+1), :]
        z = self.encoder(z)
        cls_out = z[:, 0, :]
        return self.head(cls_out).squeeze(-1)


In [17]:
# Cell 7 — Training utilities (metrics + early stopping + threshold tuning)

@torch.no_grad()
def predict_logits(model, loader):
    model.eval()
    all_logits = []
    all_y = []
    for xb, yb in loader:
        xb = xb.to(device)
        logits = model(xb)
        all_logits.append(logits.detach().cpu())
        all_y.append(yb.detach().cpu())
    logits = torch.cat(all_logits).numpy()
    y = torch.cat(all_y).numpy().astype(int)
    return logits, y

def find_best_threshold(y_true, prob, metric="f1"):
    # scan thresholds to optimize F1 by default (small dataset)
    thresholds = np.linspace(0.05, 0.95, 181)
    best_t, best_score = 0.5, -1.0
    for t in thresholds:
        pred = (prob >= t).astype(int)
        p, r, f1, _ = precision_recall_fscore_support(y_true, pred, average="binary", zero_division=0)
        score = f1 if metric == "f1" else (p + r) / 2.0
        if score > best_score:
            best_score, best_t = score, t
    return best_t, best_score

def compute_metrics(y_true, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    acc = accuracy_score(y_true, pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, pred, average="binary", zero_division=0)
    roc = roc_auc_score(y_true, prob) if len(np.unique(y_true)) > 1 else float("nan")
    ap  = average_precision_score(y_true, prob) if len(np.unique(y_true)) > 1 else float("nan")
    cm = confusion_matrix(y_true, pred)
    return {"acc": acc, "precision": p, "recall": r, "f1": f1, "roc_auc": roc, "pr_auc": ap, "cm": cm}

def train_model(
    model,
    train_loader,
    val_loader,
    epochs=60,
    lr=1e-3,
    weight_decay=1e-4,
    patience=10,
):
    model = model.to(device)

    # pos_weight cho BCEWithLogitsLoss: (Nneg / Npos)
    counts = np.bincount(y_train)
    pos_weight = torch.tensor([counts[0] / max(counts[1], 1)], dtype=torch.float32, device=device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val = -1.0
    best_state = None
    bad = 0

    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        n = 0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item() * xb.size(0)
            n += xb.size(0)

        # validation
        val_logits, val_y = predict_logits(model, val_loader)
        val_prob = 1.0 / (1.0 + np.exp(-val_logits))

        # dùng PR-AUC làm early-stopping metric (ổn khi imbalance)
        val_ap = average_precision_score(val_y, val_prob) if len(np.unique(val_y)) > 1 else 0.0

        if val_ap > best_val:
            best_val = val_ap
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1

        if ep % 5 == 0 or ep == 1:
            t, f1s = find_best_threshold(val_y, val_prob, metric="f1")
            m = compute_metrics(val_y, val_prob, threshold=t)
            print(f"epoch {ep:03d} | loss {total_loss/n:.4f} | val PR-AUC {val_ap:.4f} | val F1 {m['f1']:.4f} @thr {t:.2f}")

        if bad >= patience:
            print(f"Early stopping at epoch {ep}, best val PR-AUC={best_val:.4f}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


In [18]:
# Cell 8 — Pick a model, train, tune threshold on val, evaluate on test

MODEL_TYPE = "mlp"   # "mlp" | "cnn" | "transformer"
d_in = X_train_s.shape[1]

if MODEL_TYPE == "mlp":
    model = MLP(d_in=d_in, hidden=(256, 128, 64), dropout=0.25)
elif MODEL_TYPE == "cnn":
    model = CNN1D(d_in=d_in, channels=(32, 64, 128), dropout=0.25)
elif MODEL_TYPE == "transformer":
    model = TabTransformer(n_features=d_in, d_model=128, nhead=8, nlayers=2, dropout=0.1)
else:
    raise ValueError("Unknown MODEL_TYPE")

model = train_model(
    model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=80,
    lr=1e-3,
    weight_decay=1e-4,
    patience=12,
)

# tune threshold on val
val_logits, val_y = predict_logits(model, val_loader)
val_prob = 1.0 / (1.0 + np.exp(-val_logits))
best_thr, _ = find_best_threshold(val_y, val_prob, metric="f1")
print("best threshold (val) =", best_thr)

# evaluate on test
test_logits, test_y = predict_logits(model, test_loader)
test_prob = 1.0 / (1.0 + np.exp(-test_logits))
metrics = compute_metrics(test_y, test_prob, threshold=best_thr)

print("\nTEST METRICS:")
for k in ["acc", "precision", "recall", "f1", "roc_auc", "pr_auc"]:
    print(f"{k:>10s} = {metrics[k]:.4f}")
print("\nConfusion matrix [[TN FP],[FN TP]]:\n", metrics["cm"])


epoch 001 | loss 1.0796 | val PR-AUC 0.2609 | val F1 0.3879 @thr 0.49
epoch 005 | loss 0.4859 | val PR-AUC 0.2446 | val F1 0.3605 @thr 0.36
epoch 010 | loss 0.3559 | val PR-AUC 0.2265 | val F1 0.3431 @thr 0.07
Early stopping at epoch 13, best val PR-AUC=0.2609
best threshold (val) = 0.48999999999999994

TEST METRICS:
       acc = 0.5646
 precision = 0.4112
    recall = 0.8224
        f1 = 0.5483
   roc_auc = 0.7031
    pr_auc = 0.4494

Confusion matrix [[TN FP],[FN TP]]:
 [[100 126]
 [ 19  88]]


In [19]:
# Cell 9 (optional) — Save model + scaler for later inference

SAVE_DIR = "./tab_dl_checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

torch.save(model.state_dict(), os.path.join(SAVE_DIR, f"{MODEL_TYPE}_state.pt"))

import joblib
joblib.dump(scaler, os.path.join(SAVE_DIR, "scaler.joblib"))

meta = {
    "feature_cols": feature_cols,
    "label_col": LABEL_COL,
    "group_col": GROUP_COL,
    "best_threshold": float(best_thr),
    "model_type": MODEL_TYPE,
    "input_dim": int(d_in),
}
joblib.dump(meta, os.path.join(SAVE_DIR, "meta.joblib"))

print("Saved to:", SAVE_DIR)
print(meta)


Saved to: ./tab_dl_checkpoints
{'feature_cols': ['ast_n_nodes', 'ast_n_edges', 'ast_n_leaves', 'ast_max_depth', 'ast_avg_branching', 'ast_error_nodes', 'ast_truncated', 'parse_ok', 'feat_0', 'feat_1', 'feat_2', 'feat_3', 'feat_4', 'feat_5', 'feat_6', 'feat_7', 'feat_8', 'feat_9', 'feat_10', 'feat_11', 'feat_12', 'feat_13', 'feat_14', 'feat_15', 'feat_16', 'feat_17', 'feat_18', 'feat_19', 'feat_20', 'feat_21', 'feat_22', 'feat_23', 'feat_24', 'feat_25', 'feat_26', 'feat_27', 'feat_28', 'feat_29', 'feat_30', 'feat_31', 'feat_32', 'feat_33', 'feat_34', 'feat_35', 'feat_36', 'feat_37', 'feat_38', 'feat_39', 'feat_40', 'feat_41', 'feat_42', 'feat_43', 'feat_44', 'feat_45', 'feat_46', 'feat_47', 'feat_48', 'feat_49', 'feat_50', 'feat_51', 'feat_52', 'feat_53', 'feat_54', 'feat_55', 'feat_56', 'feat_57', 'feat_58', 'feat_59', 'feat_60', 'feat_61', 'feat_62', 'feat_63', 'feat_64', 'feat_65', 'feat_66', 'feat_67', 'feat_68', 'feat_69', 'feat_70', 'feat_71', 'feat_72', 'feat_73', 'feat_74', 'fea